In [74]:
import os
import numpy as np
import pandas as pd
from glob import glob
from mne.decoding import CSP
from sklearn.preprocessing import StandardScaler

In [75]:
pd.read_csv("dataset/derivative_mid_5min/sub-001.csv")

,Fp1,Fp2,F3,F4,C3,C4,P3,P4,O1,O2,F7,F8,T3,T4,T5,T6,Fz,Cz,Pz
0,28.106768,27.441973,23.673218,19.875109,18.152576,15.297676,18.360073,16.858160,15.213113,22.900726,23.686169,25.682659,22.683741,29.406986,16.644232,19.460236,19.185287,16.738230,17.906288
1,26.727509,26.027483,22.766195,18.799307,17.922928,14.887482,18.263588,17.043249,15.528177,23.191498,23.063129,25.104593,22.456659,28.306454,17.296949,19.315311,18.255154,16.482952,17.974981
2,25.426458,24.821260,21.670372,18.061966,17.749981,14.876583,18.253233,17.359100,15.895592,23.393311,22.509563,24.801735,22.091560,27.072908,17.793692,19.173122,17.506075,16.362591,18.104002
3,24.698235,24.245667,20.612814,17.946365,17.708075,15.343101,18.392147,17.848988,16.297136,23.588772,22.261555,24.921106,21.722656,26.061998,18.111496,19.207611,17.145636,16.513475,18.382906
4,24.880493,24.566380,19.826670,18.596111,17.852354,16.241646,18.706148,18.490246,16.699085,23.836700,22.469017,25.498295,21.471867,25.558849,18.246462,19.519907,17.319199,17.028008,18.853109
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149995,-29.882614,-27.946108,-25.226284,-30.676615,-40.321236,-35.036724,-47.252869,-37.219330,-45.141586,-31.135717,-34.031631,-32.407887,-36.358513,-22.613909,-45.715069,-37.480389,-29.165955,-40.390945,-40.866695
149996,-29.748585,-28.737083,-25.877140,-31.508936,-40.898178,-36.616257,-48.181320,-38.696621,-45.028839,-32.020386,-34.361858,-34.101295,-36.252682,-24.010927,-45.754990,-38.761135,-30.972271,-41.958172,-42.008701
149997,-29.006845,-29.226044,-27.008053,-32.265182,-41.248318,-38.000496,-48.705769,-39.919479,-44.066555,-32.392872,-34.488945,-35.917274,-35.905785,-25.476727,-44.781517,-39.716640,-32.501057,-42.966393,-42.754986
149998,-27.817333,-29.374304,-28.460005,-32.965008,-41.502724,-39.044086,-48.935513,-40.722713,-42.644733,-32.286770,-34.441036,-37.503899,-35.536884,-26.672396,-43.218674,-40.188229,-33.591808,-43.367355,-43.063515


In [76]:
# read the paticipation data
participants_file = "dataset/participants.tsv"
participants_df = pd.read_csv(participants_file, sep='\t')
participants_df = participants_df.drop(columns=['Unnamed: 0'])

participants_df.head()

,participant_id,Gender,Age,Group,MMSE,Set
0,sub-023,M,60,A,16,Train
1,sub-021,M,79,A,22,Train
2,sub-003,M,70,A,14,Train
3,sub-020,M,71,A,4,Train
4,sub-012,M,63,A,18,Train


In [77]:
# read eeg data
eeg_files = glob("dataset/derivative_mid_5min/*.csv")

label_map = {"A":0, "F": 1, "C": 2}

all_eeg_data = []
all_eeg_labels = []

for file in eeg_files:
    sub_id = os.path.basename(file).split('.')[0]
    print("reading file: ", sub_id)

    df = pd.read_csv(file)

    eeg_data = df.values[:, 1:]
    eeg_data = np.transpose(eeg_data)  # change to (n_channels, n_samples)
    eeg_data = np.expand_dims(eeg_data, axis=0)  # change to 3D (1, n_channels, n_samples)

    patient_info = participants_df[participants_df['participant_id'] == sub_id]
    
    # standardize the data
    scaler = StandardScaler()
    eeg_data = scaler.fit_transform(eeg_data.squeeze().T).T 
    eeg_data = np.expand_dims(eeg_data, axis=0)

    group_label = patient_info['Group'].values[0]
    label = label_map[group_label]
    labels = np.full(eeg_data.shape[0], label)

    all_eeg_data.append(eeg_data)
    all_eeg_labels.append(labels)

reading file:  sub-001
reading file:  sub-002
reading file:  sub-003
reading file:  sub-004
reading file:  sub-005
reading file:  sub-006
reading file:  sub-007
reading file:  sub-008
reading file:  sub-009
reading file:  sub-010
reading file:  sub-011
reading file:  sub-012
reading file:  sub-013
reading file:  sub-014
reading file:  sub-015
reading file:  sub-016
reading file:  sub-017
reading file:  sub-018
reading file:  sub-019
reading file:  sub-020
reading file:  sub-021
reading file:  sub-022
reading file:  sub-023
reading file:  sub-024
reading file:  sub-025
reading file:  sub-026
reading file:  sub-027
reading file:  sub-028
reading file:  sub-029
reading file:  sub-030
reading file:  sub-031
reading file:  sub-032
reading file:  sub-033
reading file:  sub-034
reading file:  sub-035
reading file:  sub-036
reading file:  sub-037
reading file:  sub-038
reading file:  sub-039
reading file:  sub-040
reading file:  sub-041
reading file:  sub-042
reading file:  sub-043
reading fil

In [78]:
X = np.concatenate(all_eeg_data, axis=0)
y = np.array(all_eeg_labels).flatten()

In [79]:
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

X shape: (88, 18, 150000)
y shape: (88,)


In [80]:
# calculate the CSP features
csp = CSP(n_components=4, reg=None, log=True)
filtered_eeg = csp.fit_transform(X, y)

filtered_df = pd.DataFrame(filtered_eeg, columns=[f'CSP_{i+1}' for i in range(filtered_eeg.shape[1])])

Computing rank from data with rank=None
    Using tolerance 59 (2.2e-16 eps * 18 dim * 1.5e+16  max singular value)
    Estimated rank (data): 18
    data: rank 18 computed from 18 data channels with 0 projectors
Reducing data rank from 18 -> 18
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.


In [ ]:
# Load participant information
participants_file = "dataset/participants.tsv"
participants_df = pd.read_csv(participants_file, sep='\t')

# Load EEG data files
eeg_files = glob("dataset/derivative_mid_5min/*.csv")

label_map = {"A": 0, "F": 1, "C": 2}  # Class mapping
all_segments = []  # Store all time segments
all_labels = []  # Store all EEG segment labels

# Define time window (number of samples)
fs = 500  # Sampling rate 500Hz
window_size = int(0.2 * fs)
segments_range = [(0, 4), (9, 11)]  # Select data from 0-4s and 9-11s


In [ ]:
# Iterate over each EEG file
for file in eeg_files:
    sub_id = os.path.basename(file).split('.')[0]

    df = pd.read_csv(file)
    eeg_data = df.values 
    eeg_data = np.transpose(eeg_data)  # Reshape to (n_channels, n_samples)

    # Retrieve participant information
    patient_info = participants_df[participants_df['participant_id'] == sub_id]

    # Normalize EEG data
    scaler = StandardScaler()
    eeg_data = scaler.fit_transform(eeg_data)

    # Compute label
    group_label = patient_info['Group'].values[0]
    label = label_map[group_label]

    # Store all segments of this EEG data
    file_segments = []

    # Extract segments from 0-4s and 9-11s
    for start, end in segments_range:
        for t in np.arange(start, end, 0.2):
            start_idx = int(t * fs)
            end_idx = start_idx + window_size
            if end_idx <= eeg_data.shape[1]:
                segment = eeg_data[:, start_idx:end_idx]  # Extract (n_channels, window_size) segment
                file_segments.append(segment)

    if len(file_segments) == 30:  # Ensure each trial generates 30 segments
        all_segments.append(file_segments)
        all_labels.append(label)

In [83]:
# Convert to NumPy arrays
all_segments = np.array(all_segments)  # Shape: (n_trials, 30, n_channels, 100)
all_labels = np.array(all_labels)  # Shape: (n_trials,)

In [ ]:
# Compute CSP and concatenate features
final_features = []
for i in range(30):
    csp = CSP(n_components=1, reg=None, log=True)
    filtered_eeg_i = csp.fit_transform(all_segments[:, i, :, :], all_labels)
    final_features.append(filtered_eeg_i.flatten())

Computing rank from data with rank=None
    Using tolerance 0.9 (2.2e-16 eps * 19 dim * 2.1e+14  max singular value)
    Estimated rank (data): 18
    data: rank 18 computed from 19 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 19 -> 18
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 0.91 (2.2e-16 eps * 19 dim * 2.1e+14  max singular value)
    Estimated rank (data): 18
    data: rank 18 computed from 19 data channels with 0 projectors
    Setting small data eigenvalues to zero (without PCA)
Reducing data rank from 19 -> 18
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Estimating class=2 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 0.92 (2.2e-16 eps *

In [85]:
# Concatenate all CSP features from time windows
final_features = np.array(final_features).T  # Transpose to shape (n_trials, 30)

# Convert to DataFrame
filtered_df = pd.DataFrame(final_features, columns=[f'CSP_{i+1}' for i in range(30)])
filtered_df['Group'] = all_labels  # Add class labels

# Save processed data
filtered_df.to_csv("dataset/csp_filtered_eeg_data.csv", index=False)

In [86]:
filtered_df

,CSP_1,CSP_2,CSP_3,CSP_4,CSP_5,CSP_6,CSP_7,CSP_8,CSP_9,CSP_10,...,CSP_22,CSP_23,CSP_24,CSP_25,CSP_26,CSP_27,CSP_28,CSP_29,CSP_30,Group
0,-0.777756,-1.644552,-0.537392,0.067447,-1.158560,-0.593630,-1.097554,0.298367,-0.851302,-0.909989,...,0.430412,-0.090268,0.824146,-0.766176,-0.861832,-1.886289,-0.534093,-0.380193,-0.842999,0
1,-0.485548,0.059792,-0.233942,-0.324358,-1.303932,-0.953520,-0.672807,0.040761,-0.795979,-0.671857,...,-0.164277,-1.230120,-0.188800,-0.087393,-0.677838,0.289758,-1.204512,-0.879613,-0.353986,0
2,-0.533171,-1.988423,0.034893,-0.169467,-0.233121,-0.381867,-0.521115,-0.311176,-0.179104,-1.345807,...,-0.962197,-0.834184,0.086984,-1.796654,-0.034531,-1.411410,-1.272214,0.053737,0.512052,0
3,-2.477917,-0.596295,2.820875,-2.405341,-3.790458,-3.163147,-2.086040,2.030756,-2.678488,2.785722,...,-1.134210,0.147608,-1.063420,-1.565861,0.720743,-1.035529,-1.058317,-0.831014,-1.936618,0
4,0.474048,1.168248,1.926979,0.599176,0.469829,-1.517959,0.415533,-0.376823,-1.662290,0.140203,...,1.332648,0.752185,2.226301,-1.025025,2.422548,1.645842,0.315085,1.140624,-1.223041,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83,-1.107460,2.598071,-2.337370,2.195434,-1.004265,-0.415631,0.839094,-1.379340,-1.692309,-1.032192,...,-0.435386,-0.070443,-2.252464,0.774878,1.324787,1.851356,0.447253,-0.915148,-0.501469,1
84,-2.817878,0.488486,-2.477823,-0.748489,0.171236,-1.430643,-1.010020,-0.279660,-0.346102,-1.976000,...,0.329394,-1.138251,-2.404561,-2.665251,-0.368557,-0.285537,-0.001489,-0.760167,-1.247734,1
85,-2.467302,1.509222,-2.924102,0.938738,0.415136,0.215985,2.691829,-3.409968,-0.310291,-2.357459,...,2.778145,1.239521,-2.588956,1.101072,1.189532,1.597050,2.327214,-2.389675,-2.682156,1
86,-1.134349,-0.204780,-0.885030,0.043495,-0.015400,-1.362952,-0.806578,-1.307602,-0.258213,-0.030280,...,0.654957,-0.775567,-0.612858,0.484745,-0.256033,-1.722956,-1.277242,-2.430273,-0.647316,1
